In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [0]:
emp=spark.createDataFrame(emp_data,schema=emp_schema)
display(emp)

In [0]:
emp.printSchema()

In [0]:
emp_gender_fix=emp.withColumn("new_gender",when(emp["gender"]=="Male","M").when(emp["gender"]=="Female","F").otherwise("Other"))
display(emp_gender_fix)

In [0]:
emp_rename=emp.withColumn("new_name",regexp_replace(emp["name"],"M","R"))
emp_rename.show()

In [0]:
emp_date_fix=emp.withColumn("new_hire_DATE",to_date(emp["hire_date"],"yyyy-MM-dd"))
emp_date_fix.printSchema()

In [0]:
emp_dated=emp_date_fix.withColumn("current_date",current_date()).withColumn("current_time",current_timestamp())
display(emp_dated)

In [0]:
# Drop Null gender records
emp_1 = emp_dated.na.drop()
emp_1.show()

In [0]:
# Create DataFrame for region A
df_A = spark.createDataFrame([("apple", 3, 5), ("banana", 1, 10), ("orange", 2, 8)], ["Name", "Col_1", "Col_2"])
df_A.show()
# Create DataFrame for region B
df_B = spark.createDataFrame([("apple", 3, 5), ("banana", 1, 15), ("grape", 4, 6)], ["Name", "Col_1", "Col_3"])
df_B.show()

In [0]:
df_A.union(df_B).show()

In [0]:
data = [(1, 1), (2, 4), (3, 9), (4, 16), (5, 25)]
df = spark.createDataFrame(data, ["actual", "predicted"])
df.show()

In [0]:
df = df.withColumn("squared_error", pow((col("actual") - col("predicted")), 2))
df.show()

In [0]:
mse=df.agg({"squared_error": "avg"}).collect()[0][0]
print(mse)

In [0]:
emp.summary().show()

In [0]:
from pyspark.sql import functions as F
df = emp.withColumn('name_length', F.length(emp.name))
df.show()

How to compute difference of differences between consecutive numbers of a column?

In [0]:
# For the sake of example, we'll create a sample DataFrame
data = [('James', 34, 55000),
('Michael', 30, 70000),
('Robert', 37, 60000),
('Maria', 29, 80000),
('Jen', 32, 65000)]
df = spark.createDataFrame(data, ["name", "age" , "salary"])
df.show()

In [0]:
data_new=df.withColumn("previous_salary",lag(df["salary"],1).over(Window.orderBy(df["salary"]))).withColumn("differnce",df["salary"]-col("previous_salary")).drop("previous_salary")
data_new.show()

21. How to get the day of month, week number, day of year and day of week from a date strings?


In [0]:
data = [("2023-05-18","01 Jan 2010",), ("2023-12-31", "01 Jan 2010",)]
df = spark.createDataFrame(data, ["date_str_1", "date_str_2"])
df.show()

In [0]:
df=df.withColumn("date_1",to_date(df["date_str_1"],'yyyy-MM-dd')).withColumn("date_2",to_date(df["date_str_2"],'dd MMM yyyy'))
df.show()
df.printSchema()

In [0]:
df = df.withColumn("date_3", to_date(df.date_str_1, 'yyyy-MM-dd'))
df = df.withColumn("date_4", to_date(df.date_str_2, 'dd MMM yyyy'))
df.show()

In [0]:
df = df.withColumn("day_of_month", dayofmonth(df.date_1))\
.withColumn("week_number", weekofyear(df.date_1))\
.withColumn("day_of_year", dayofyear(df.date_1))\
.withColumn("day_of_week", dayofweek(df.date_1))
df.show()

In [0]:
df = spark.createDataFrame([('Jan 2010',), ('Feb 2011',), ('Mar 2012',)], ['MonthYear'])
df.show()

In [0]:
df = df.withColumn('Date', expr("to_date(MonthYear, 'MMM yyyy')"))
df.show()# replace day with 4

df = df.withColumn('Date', expr("date_add(date_sub(Date, day(Date) - 1), 3)"))
df2=df.withColumn('Date5', expr("date_sub(Date,day(Date2) - 1)"))
df2.show()